# Integrating transcriptomics data (eFlux, GIMME, iMAT)

This notebook shows how to turn a gene expression profile into condition-specific flux predictions and context-specific models with MEWpy's three omics integration methods:

| Method | Idea | Reference |
|---|---|---|
| **eFlux** | reaction bounds are scaled by the (normalised) expression of the reaction | Colijn et al. 2009, PLoS Comput Biol 5(8):e1000489 |
| **GIMME** | keep a fraction of the maximal growth while minimising flux through lowly expressed reactions | Becker & Palsson 2008, PLoS Comput Biol 4(5):e1000082 |
| **iMAT** | MILP that maximises the number of reactions whose activity agrees with high/low expression | Shlomi et al. 2008, Nat Biotechnol 26:1003 |

**Data.** The *S. cerevisiae* genome-scale model iMM904 and a microarray compendium (3700 genes x 247 samples, ArrayExpress series E-MEXP-130, E-MEXP-26, ...), both shipped in `examples/models/germ/` (the same expression file is used by the CoRegFlux example in the GERM documentation).

Every method below accepts any MEWpy simulator, so the same code works with COBRApy and REFRAMED models.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import cobra
import pandas as pd

from mewpy.simulation import get_simulator
from mewpy.omics import ExpressionSet, Preprocessing, eFlux, GIMME, iMAT

model = cobra.io.read_sbml_model('models/germ/iMM904.xml')
sim = get_simulator(model)
BIOMASS = 'BIOMASS_SC5_notrace'
wild_type = sim.simulate()
print('reactions:', len(sim.reactions), '| genes:', len(sim.genes))
print('FBA growth rate:', round(wild_type.objective_value, 4))

2026-11-09 18:53:59,934	INFO util.py:155 -- Outdated packages:
  ipywidgets==7.8.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


reactions: 1577 | genes: 905
FBA growth rate: 0.2879


## 1. Load the expression data

`ExpressionSet` holds a genes x conditions matrix. Here we read the CSV with pandas and keep a few samples. Gene identifiers must match the model's gene ids (here ORF names such as `YMR056C`).

In [2]:
data = pd.read_csv('models/germ/iMM904_gene_expression.csv', sep=';', index_col=0)
print('data:', data.shape, '| value range: %.2f to %.2f' % (data.values.min(), data.values.max()))

overlap = set(data.index) & set(sim.genes)
print('model genes with expression values: %d of %d' % (len(overlap), len(sim.genes)))
data.iloc[:5, :4]

data: (3700, 247) | value range: 3.77 to 14.70
model genes with expression values: 829 of 905


,E.MEXP.130_1,E.MEXP.130_2,E.MEXP.130_3,E.MEXP.130_4
Gene,,,,
YMR056C,9.551500,9.492630,9.33030,9.351230
YBR085W,7.532370,7.307645,7.51121,7.460775
YJR155W,8.544395,8.594950,8.85956,8.906815
YNL331C,9.271940,9.349005,9.66355,9.471625
YDL243C,5.817845,5.590550,5.87841,5.664595


The values range from about 3.8 to 14.7, i.e. they are on a log2 scale. eFlux scales reaction bounds by expression and therefore needs **linear-scale, non-negative** values (it raises a `ValueError` on negative data), so we convert the samples used below back with `2 ** x`.

In [3]:
conditions = list(data.columns[:3])
expr = ExpressionSet.from_dataframe(2 ** data[conditions])
condition = conditions[0]
print('conditions:', conditions)

conditions: ['E.MEXP.130_1', 'E.MEXP.130_2', 'E.MEXP.130_3']


## 2. From gene to reaction expression

All three methods first map gene expression onto reactions through the GPR rules: by default an `AND` (enzyme complex) takes the **minimum** and an `OR` (isozymes) the **maximum** of its genes. Reactions without a GPR, or whose genes have no measurement, get no value: eFlux gives them the maximum relative capacity (1), and GIMME/iMAT neither penalise nor classify them.

In [4]:
rxn_exp = Preprocessing(sim, expr).reactions_expression(condition)
print('reactions with an expression value: %d of %d' % (len(rxn_exp), len(sim.reactions)))
pd.Series(rxn_exp).describe()

reactions with an expression value: 1010 of 1577


count     1010.000000
mean      1927.731466
std       2678.264931
min         27.951040
25%        382.855215
50%        908.788714
75%       2372.759841
max      20867.938265
dtype: float64

## 3. eFlux

Each reaction bound becomes its expression divided by the maximum expression, so the objective value is on a relative scale. Pass `scale_rxn`/`scale_value` to rescale the fluxes to a measured rate (here a glucose uptake of 10 mmol/gDW/h).

In [5]:
sol_eflux = eFlux(sim, expr, condition=condition, scale_rxn='EX_glc__D_e', scale_value=10)
print('status:', sol_eflux.status)
print('growth rate (rescaled to 10 mmol/gDW/h glucose): %.4f' % sol_eflux.fluxes[BIOMASS])

status: OPTIMAL
growth rate (rescaled to 10 mmol/gDW/h glucose): 0.1312


## 4. GIMME

Reactions whose expression is below the `cutoff` percentile are penalised in proportion to how far they are below it, while growth is kept at `growth_frac` of its maximum. The objective value is the inconsistency score (0 = no lowly expressed reaction carries flux). The growth rate is therefore `growth_frac` x the FBA optimum by construction.

In [6]:
sol_gimme = GIMME(sim, expr, condition=condition, cutoff=25, growth_frac=0.9)
print('status:', sol_gimme.status, '| inconsistency score: %.4f' % sol_gimme.objective_value)
print('growth rate: %.4f' % sol_gimme.fluxes[BIOMASS])

status: OPTIMAL | inconsistency score: 212.0336
growth rate: 0.2591


## 5. iMAT

Reactions above the upper percentile are *highly* expressed and those below the lower percentile *lowly* expressed. iMAT finds a flux distribution in which as many highly expressed reactions as possible carry flux (|v| >= `epsilon`) and as many lowly expressed ones as possible carry none. The objective counts those agreements. iMAT has no growth requirement unless `growth_frac` is given (without it the best agreement may have zero growth), so we require 90% of the maximal growth here.

In [7]:
sol_imat = iMAT(sim, expr, condition=condition, cutoff=(25, 75), epsilon=1, growth_frac=0.9)
print('status:', sol_imat.status, '| agreement (objective): %.0f' % sol_imat.objective_value)
print('growth rate: %.4f' % sol_imat.fluxes[BIOMASS])

status: OPTIMAL | agreement (objective): 310
growth rate: 0.2591


## 6. Comparing the predicted fluxes

GIMME and iMAT only score which reactions carry flux, so their flux values are not unique: reactions outside the scored set can take any value, including loops that run up to the model bounds (±999999 in this COBRApy model). For flux comparisons use `GIMME(..., parsimonious=True)`, which then minimises total flux; iMAT's result is its on/off pattern, best used through the context-specific model (section 7).

In [8]:
sol_gimme_p = GIMME(sim, expr, condition=condition, cutoff=25, growth_frac=0.9, parsimonious=True)
reactions = [BIOMASS, 'EX_glc__D_e', 'EX_o2_e', 'EX_etoh_e', 'EX_co2_e', 'PGI', 'PFK', 'PYK', 'CSm', 'ICDHxm']
pd.DataFrame({
    'FBA': {r: wild_type.fluxes[r] for r in reactions},
    'eFlux': {r: sol_eflux.fluxes[r] for r in reactions},
    'GIMME (parsimonious)': {r: sol_gimme_p.fluxes[r] for r in reactions},
}).round(3)

,FBA,eFlux,GIMME (parsimonious)
BIOMASS_SC5_notrace,0.288,0.131,0.259
EX_glc__D_e,-10.000,-10.000,-8.985
EX_o2_e,-2.000,-0.263,-2.000
EX_etoh_e,15.815,15.651,11.487
EX_co2_e,18.021,17.009,15.841
PGI,0.000,0.000,7.707
PFK,0.000,3.439,7.704
PYK,17.722,16.412,15.882
CSm,0.529,0.134,0.264
ICDHxm,0.000,0.000,0.000


## 7. Building a context-specific model

With `build_model=True` each method also returns a simulator of a context-specific model that can be simulated, analysed further or saved as SBML:

- **GIMME** removes the reactions that are neither above the expression cutoff nor used by its solution; it works on an irreversible copy, so reversible reactions appear split into `_REV` halves;
- **iMAT** keeps the objective reactions and every reaction that carries flux in its solution; when building it requires 90% of the wild-type growth by default (`growth_frac`), so the model can grow;
- **eFlux** keeps the expression-scaled bounds on the model and removes reactions with negligible capacity, so the built model's FBA objective is the (unscaled, relative) eFlux objective.

The original model is never modified.

In [9]:
import os, tempfile

out_dir = tempfile.mkdtemp()
rows = {}
for name, method in [('eFlux', eFlux), ('GIMME', GIMME), ('iMAT', iMAT)]:
    _, context_sim = method(sim, expr, condition=condition, build_model=True)
    result = context_sim.simulate()
    path = os.path.join(out_dir, f'iMM904_{name}.xml')
    cobra.io.write_sbml_model(context_sim.model, path)
    rows[name] = {'reactions': len(context_sim.reactions), 'status': str(result.status),
                  'objective': result.objective_value, 'sbml': os.path.basename(path)}
pd.DataFrame(rows).T

Converting to irreversible:   0%|          | 0/1577 [00:00<?, ?it/s]

Converting to irreversible:  13%|█▎        | 203/1577 [00:00<00:00, 2026.98it/s]

Converting to irreversible:  26%|██▌       | 409/1577 [00:00<00:00, 2017.14it/s]

Converting to irreversible:  44%|████▍     | 696/1577 [00:00<00:00, 2378.52it/s]

Converting to irreversible:  59%|█████▉    | 934/1577 [00:00<00:00, 2041.79it/s]

Converting to irreversible:  73%|███████▎  | 1144/1577 [00:00<00:00, 2010.22it/s]

Converting to irreversible:  86%|████████▌ | 1349/1577 [00:00<00:00, 2013.60it/s]

Converting to irreversible:  98%|█████████▊| 1553/1577 [00:00<00:00, 1792.44it/s]

Converting to irreversible: 100%|██████████| 1577/1577 [00:00<00:00, 1934.85it/s]

,reactions,status,objective,sbml
eFlux,1577,OPTIMAL,0.003395,iMM904_eFlux.xml
GIMME,1021,OPTIMAL,0.286233,iMM904_GIMME.xml
iMAT,406,OPTIMAL,0.287866,iMM904_iMAT.xml


The saved SBML files are plain COBRA models: `cobra.io.read_sbml_model(path)` loads them back for use in any other tool. The original model is unchanged:

In [10]:
print('original model reactions:', len(model.reactions), '| growth: %.4f' % sim.simulate().objective_value)

original model reactions: 1577 | growth: 0.2879


## 8. Same analysis with a REFRAMED model

MEWpy simulators behave the same for COBRApy and REFRAMED models. REFRAMED keeps the SBML prefixes (`G_YMR056C`, `R_PGI`), and MEWpy matches the expression identifiers to the prefixed gene ids automatically, so the same `ExpressionSet` can be used:

In [11]:
from reframed.io.sbml import load_cbmodel

rsim = get_simulator(load_cbmodel('models/germ/iMM904.xml'))
r_bio = 'R_' + BIOMASS
r_gimme = GIMME(rsim, expr, condition=condition, cutoff=25, growth_frac=0.9)
r_imat = iMAT(rsim, expr, condition=condition, cutoff=(25, 75), epsilon=1, growth_frac=0.9)
pd.DataFrame({
    'COBRApy': {'GIMME score': sol_gimme.objective_value, 'GIMME growth': sol_gimme.fluxes[BIOMASS],
                'iMAT agreement': sol_imat.objective_value},
    'REFRAMED': {'GIMME score': r_gimme.objective_value, 'GIMME growth': r_gimme.fluxes[r_bio],
                 'iMAT agreement': r_imat.objective_value},
}).round(4)

,COBRApy,REFRAMED
GIMME score,212.0336,212.0336
GIMME growth,0.2591,0.2591
iMAT agreement,310.0000,310.0000


## 9. Several conditions

Each method takes a `condition` (a column name or index), so comparing samples is a loop. Here we take one sample from each of three ArrayExpress series. GIMME's growth is fixed by `growth_frac`, so the informative values are eFlux's (expression-limited) growth and the GIMME/iMAT scores:

In [12]:
first_of_series = {}
for column in data.columns:
    first_of_series.setdefault(column.rsplit('_', 1)[0], column)
samples = list(first_of_series.values())[:3]
expr_s = ExpressionSet.from_dataframe(2 ** data[samples])
pd.DataFrame({s: {
    'eFlux growth': eFlux(sim, expr_s, condition=s, scale_rxn='EX_glc__D_e', scale_value=10).fluxes[BIOMASS],
    'GIMME score': GIMME(sim, expr_s, condition=s).objective_value,
    'iMAT agreement': iMAT(sim, expr_s, condition=s, growth_frac=0.9).objective_value,
} for s in samples}).T.round(4)

,eFlux growth,GIMME score,iMAT agreement
E.MEXP.130_1,0.1312,212.0336,310.0
E.MEXP.26_1,0.1064,395.3242,314.0
E.MEXP.27_1,0.0711,682.3040,316.0


## Notes

- **Thresholds matter.** GIMME's `cutoff` and iMAT's `cutoff=(low, high)` are percentiles of the reaction expression in the chosen condition; results can change considerably with them.
- **Scale.** Use linear-scale values for eFlux. `ExpressionSet` also offers preprocessing helpers (`apply`, `quantile_pipeline`).
- **Missing values.** Genes with NaN are treated as unmeasured.
- **Non-unique fluxes.** GIMME and iMAT constrain which reactions are active, not flux values; compare fluxes only after a parsimonious step or within the context-specific model.
- **Solver.** iMAT is a MILP and is the slowest of the three on large models; CPLEX or Gurobi are faster than SCIP for large instances.